# 🧠 Smart MCQ Solver — V2 Improved Pipeline (Target: 0.80+ MAP@3)

**Roll No**: 24f1002384 | **Notebook**: `DL-24f1002384-notebook-t22026`

## What changed from V1 based on log analysis:
- ❌ **Dropped DeBERTa-v3** — produced NaN loss across all folds, wasted 50 minutes with 0 contribution
- ✅ **LoRA RoBERTa is the core** — V1 already achieved **0.7688 MAP@3** in 2 epochs. Now run 3-fold CV × 3 epochs
- ✅ **Added LoRA ALBERT** as second pretrained model for ensemble diversity
- ✅ **Improved fuzzy lookup** — upgraded threshold logic using token_set_ratio for preamble-stripped matching
- ✅ **BM25 scratch model** — stronger than TF-IDF+Word2Vec baseline
- ✅ **Ensemble reweighted** — LoRA models dominate (0.80 total weight)

**Guidelines compliance**: No external APIs | 3 models (from-scratch + 2 pretrained) | W&B logging | MAP@3 metric

---

## ⚙️ Section 0 — Setup & Installation

In [ ]:
# Safe install — does not overwrite Kaggle's pre-configured PyTorch/CUDA environment
import subprocess, sys

# FIX: Remove torchao — Kaggle ships torchao 0.10.0 but newer PEFT requires >=0.16.0
# torchao is only needed for quantization, NOT for basic LoRA — safe to remove
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)

pkgs = [
    'datasets',
    'transformers>=4.40',
    'sentence-transformers',
    'faiss-cpu',
    'peft>=0.10',
    'accelerate>=0.27',
    'rapidfuzz',
    'rank_bm25',
    'wandb',
    'sentencepiece',
    'protobuf',
]
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     '--upgrade-strategy', 'only-if-needed'] + pkgs,
    check=True
)
print('All packages ready')

In [ ]:
import os, re, gc, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics.pairwise import cosine_similarity

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from peft import get_peft_model, LoraConfig, TaskType

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_DIR = '/kaggle/input/competitions/smart-mcq-solver-challenge'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHOICES = list('ABCDE')
LABEL2IDX = {c: i for i, c in enumerate(CHOICES)}
IDX2LABEL = {i: c for c, i in LABEL2IDX.items()}

print(f'🖥️  Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'🔥 GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import wandb, os

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    wandb.login(key=secrets.get_secret('WANDB_API_KEY'), relogin=True)
    WANDB_ON = True
    print('✅ W&B authenticated')
except Exception as e:
    print(f'⚠️ W&B offline: {e}')
    os.environ['WANDB_MODE'] = 'disabled'
    WANDB_ON = False

## 📊 Section 1 — Data Loading & EDA

In [ ]:
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')

print(f'Train: {train_df.shape} | Test: {test_df.shape}')
print(f'Train columns: {train_df.columns.tolist()}')
train_df.head(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Label distribution
train_df['answer'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white'
)
axes[0].set_title('Answer Label Distribution')
axes[0].set_xlabel('Option')
axes[0].set_ylabel('Count')

# Prompt length
train_df['prompt'].str.split().str.len().hist(bins=30, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Prompt Length (words)')

plt.tight_layout()
plt.show()
print(f'Average prompt length: {train_df["prompt"].str.split().str.len().mean():.1f} words')

## 🔍 Section 2 — Preamble Stripping & Fuzzy Test→Train Lookup

In [ ]:
# Strip common MCQ preambles to expose the core question text
# V1 found 491/500 test rows match training rows — this is our biggest signal!
PREAMBLES = [
    r'^Pick the best possible answer:\s*',
    r'^Select the most accurate option:\s*',
    r'^Identify the correct statement:\s*',
    r'^Choose the correct answer:\s*',
    r'^Determine the correct option:\s*',
    r'^Which of the following is correct\?\s*',
    r'\s*among the listed options\.?\s*$',
    r'\s*from the following choices\.?\s*$',
    r'\s*carefully\.?\s*$',
]

def strip_preambles(text: str) -> str:
    text = str(text).strip()
    for p in PREAMBLES:
        text = re.sub(p, '', text, flags=re.IGNORECASE).strip()
    return text

train_df['core'] = train_df['prompt'].apply(strip_preambles)
test_df['core']  = test_df['prompt'].apply(strip_preambles)

print('Sample original: ', train_df['prompt'].iloc[0][:80])
print('Sample stripped: ', train_df['core'].iloc[0][:80])

In [ ]:
# Build a high-confidence lookup table from test → train answers
# Using token_set_ratio which ignores word order differences from preambles
from rapidfuzz import fuzz, process

train_cores  = train_df['core'].tolist()
train_labels = train_df['answer'].tolist()

lookup_hard = {}  # >95 score — use directly as top prediction
lookup_soft = {}  # 80-95 score — use as a soft boost

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Fuzzy lookup'):
    res = process.extractOne(
        row['core'], train_cores, scorer=fuzz.token_set_ratio
    )
    if res:
        score, match_idx = res[1], res[2]
        matched_label = train_labels[match_idx]
        if score >= 95:
            lookup_hard[int(row['id'])] = matched_label
        elif score >= 80:
            lookup_soft[int(row['id'])] = matched_label

print(f'Hard lookup overrides (≥95 similarity): {len(lookup_hard)} / {len(test_df)}')
print(f'Soft lookup boosts (80-95 similarity):  {len(lookup_soft)} / {len(test_df)}')

## 📏 Section 3 — MAP@3 Metric

In [ ]:
def map_at_3(preds: list, labels: list) -> float:
    """
    Mean Average Precision @ 3.
    preds : list of lists e.g. [['A','C','B'], ...]
    labels: list of strings e.g. ['A', 'D', ...]
    """
    score = 0.0
    for p_list, true in zip(preds, labels):
        for rank, opt in enumerate(p_list[:3], 1):
            if opt == true:
                score += 1.0 / rank
                break
    return score / len(labels)

def scores_to_top3(scores: np.ndarray) -> list:
    """Convert 5-element score array to top-3 letter list."""
    return [IDX2LABEL[i] for i in np.argsort(scores)[::-1][:3]]

print('✅ MAP@3 metric defined')

## 🏗️ Section 4 — Model 1: BM25 Ranker (From Scratch)

In [ ]:
# BM25 is a significantly stronger retrieval baseline vs TF-IDF alone
# We use it here to score how relevant each answer option is to the question
from rank_bm25 import BM25Okapi

def tokenize_bm25(text: str) -> list:
    """Simple whitespace + lowercase tokenization."""
    return re.sub(r'[^a-z0-9\s]', ' ', str(text).lower()).split()

def score_with_bm25(row) -> np.ndarray:
    """Score all 5 options against the prompt using BM25."""
    q_tokens = tokenize_bm25(row['core'])
    corpus = [tokenize_bm25(str(row[c])) for c in CHOICES]
    bm25 = BM25Okapi(corpus)
    return np.array(bm25.get_scores(q_tokens))

# Validate on train data
bm25_preds_train = []
for _, r in tqdm(train_df.iterrows(), total=len(train_df), desc='BM25 Validation'):
    bm25_preds_train.append(scores_to_top3(score_with_bm25(r)))

m1_map3 = map_at_3(bm25_preds_train, train_df['answer'].tolist())
print(f'🎯 Model 1 (BM25 Scratch) Train MAP@3 = {m1_map3:.4f}')

# Log to W&B
run1 = wandb.init(project='24f1002384-t22026', name='model1_bm25_scratch', reinit=True)
wandb.log({'train_map3': m1_map3, 'model': 'bm25_scratch'})
wandb.finish()

# Generate test predictions
m1_test_preds = {}
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='BM25 Test Predict'):
    m1_test_preds[str(int(row['id']))] = scores_to_top3(score_with_bm25(row))

## 🤗 Section 5 — Model 2: LoRA Fine-Tuned RoBERTa (Pretrained — Core Model)

> **Why RoBERTa + LoRA?** V1 logs confirmed this combination achieved **0.7688 MAP@3** in just 2 epochs on a single 80/20 split.
> Now we run **3-fold cross validation × 3 epochs** for more stable and higher-quality predictions.

In [ ]:
ROBERTA_MODEL = 'roberta-base'
MAX_LEN = 256  # Reduced from 512 to fit in VRAM while fitting 5 options per batch
BATCH_TRAIN = 4
BATCH_EVAL  = 8

roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_MODEL)

class MCQDataset(Dataset):
    """
    For each row, creates 5 encoded pairs: (prompt, option_X).
    The model scores each pair and selects the highest-scoring option.
    """
    def __init__(self, df, tokenizer, max_len=256, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.tok = tokenizer
        self.max_len = max_len
        self.has_labels = has_labels

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Format: "Question: {core_prompt}" vs each option
        prompt = f"Question: {row['core']}"

        input_ids, attention_masks = [], []
        for c in CHOICES:
            enc = self.tok(
                prompt, f"Answer: {str(row[c])}",
                truncation='longest_first',
                max_length=self.max_len,
                padding='max_length',
                return_tensors='pt'
            )
            input_ids.append(enc['input_ids'].squeeze(0))
            attention_masks.append(enc['attention_mask'].squeeze(0))

        item = {
            'input_ids':      torch.stack(input_ids),       # (5, max_len)
            'attention_mask': torch.stack(attention_masks)  # (5, max_len)
        }
        if self.has_labels:
            item['labels'] = torch.tensor(LABEL2IDX[row['answer']], dtype=torch.long)
        return item

In [ ]:
class LoRAMCQModel(nn.Module):
    """
    LoRA-enhanced MCQ ranker.
    - Uses PEFT LoRA to fine-tune only ~0.5% of RoBERTa parameters
    - Scores each (prompt, option) pair independently, then selects top-3
    """
    def __init__(self, model_name, lora_r=16, lora_alpha=32):
        super().__init__()
        base = AutoModel.from_pretrained(model_name)
        peft_config = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=0.05,
            target_modules=['query', 'value']
        )
        self.encoder = get_peft_model(base, peft_config)
        # Do NOT enable gradient_checkpointing — it caused NaN loss in V1
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        B, N, L = input_ids.shape
        ids  = input_ids.view(B * N, L)
        mask = attention_mask.view(B * N, L)

        out = self.encoder(input_ids=ids, attention_mask=mask)
        cls = out.last_hidden_state[:, 0, :]                 # (B*N, H)
        cls = cls.to(self.classifier.weight.dtype)            # dtype safety
        logits = self.classifier(self.dropout(cls)).view(B, N)  # (B, 5)
        return logits

In [ ]:
# 3-Fold Cross-Validation for LoRA RoBERTa
# V1 log: single fold × 2 epochs → 0.7688. Now 3 folds × 3 epochs for stability.

gc.collect(); torch.cuda.empty_cache()

N_FOLDS  = 3
N_EPOCHS = 3
LR = 2e-4  # Higher than full fine-tune since only LoRA params update

folds = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
roberta_cv_scores = []
roberta_model_paths = []

run2 = wandb.init(
    project='24f1002384-t22026',
    name='model2_lora_roberta_cv',
    config={'model': ROBERTA_MODEL, 'lora_r': 16, 'lr': LR, 'epochs': N_EPOCHS, 'folds': N_FOLDS},
    reinit=True
)

for fold, (tr_idx, val_idx) in enumerate(folds.split(train_df, train_df['answer'])):
    print('='*55)
    print(f'  Fold {fold+1}/{N_FOLDS}')
    print('='*55)

    tr_df  = train_df.iloc[tr_idx]
    val_df = train_df.iloc[val_idx]

    tr_loader  = DataLoader(MCQDataset(tr_df, roberta_tokenizer, MAX_LEN), batch_size=BATCH_TRAIN, shuffle=True,  num_workers=0)
    val_loader = DataLoader(MCQDataset(val_df, roberta_tokenizer, MAX_LEN), batch_size=BATCH_EVAL,  shuffle=False, num_workers=0)

    model = LoRAMCQModel(ROBERTA_MODEL).to(DEVICE)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR, weight_decay=0.01
    )

    best_map3 = 0.0
    for epoch in range(N_EPOCHS):
        # ── Training ────────────────────────────────────────────────
        model.train()
        epoch_loss = 0.0
        for batch in tqdm(tr_loader, desc=f'Epoch {epoch+1}/{N_EPOCHS}'):
            optimizer.zero_grad()
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            lbls = batch['labels'].to(DEVICE)

            logits = model(ids, mask)
            loss = F.cross_entropy(logits, lbls)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(tr_loader)

        # ── Validation ──────────────────────────────────────────────
        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                ids  = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                logits = model(ids, mask).cpu().numpy()
                for row_logits in logits:
                    preds.append(scores_to_top3(row_logits))
                labels.extend([IDX2LABEL[l] for l in batch['labels'].numpy()])

        val_map = map_at_3(preds, labels)
        print(f'  Epoch {epoch+1} | Loss={avg_loss:.4f} | Val MAP@3={val_map:.4f}')
        wandb.log({f'fold{fold+1}_val_map3': val_map, f'fold{fold+1}_loss': avg_loss})

        if val_map > best_map3:
            best_map3 = val_map
            torch.save(model.state_dict(), f'roberta_fold{fold+1}.pt')
            print(f'  💾 Saved fold {fold+1} best model (MAP@3={best_map3:.4f})')

    roberta_cv_scores.append(best_map3)
    roberta_model_paths.append(f'roberta_fold{fold+1}.pt')
    del model; gc.collect(); torch.cuda.empty_cache()

mean_cv = np.mean(roberta_cv_scores)
print(f'\n⭐ RoBERTa CV Mean MAP@3: {mean_cv:.4f}')
wandb.log({'roberta_cv_mean_map3': mean_cv})
wandb.finish()

In [ ]:
# ── Model 2 Test Inference with TTA (M5 improvement) ──────────────
# Test-Time Augmentation: run each sample twice with different prompt templates
# Average the logits — virtually free MAP@3 improvement

TTA_TEMPLATES = [
    'Question: {q} Answer: {a}',                    # Original template
    'Answer the following MCQ carefully. {q} Answer: {a}',  # TTA augment
]

class MCQDatasetTTA(Dataset):
    """MCQ dataset supporting a custom prompt template with {q} and {a} placeholders."""
    def __init__(self, df, tokenizer, template, max_len=256, has_labels=False):
        self.df = df.reset_index(drop=True)
        self.tok = tokenizer
        self.template = template
        self.max_len = max_len
        self.has_labels = has_labels

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        core_q = str(row['core'])

        input_ids, attention_masks = [], []
        for c in CHOICES:
            text = self.template.format(q=core_q, a=str(row[c]))
            enc = self.tok(
                text,
                truncation=True,
                max_length=self.max_len,
                padding='max_length',
                return_tensors='pt'
            )
            input_ids.append(enc['input_ids'].squeeze(0))
            attention_masks.append(enc['attention_mask'].squeeze(0))

        item = {
            'input_ids':      torch.stack(input_ids),
            'attention_mask': torch.stack(attention_masks)
        }
        if self.has_labels:
            item['labels'] = torch.tensor(LABEL2IDX[row['answer']], dtype=torch.long)
        return item

# Run TTA inference: accumulate logits across folds AND templates
m2_logits = np.zeros((len(test_df), 5))
total_passes = len(roberta_model_paths) * len(TTA_TEMPLATES)

for path in roberta_model_paths:
    model_inf = LoRAMCQModel(ROBERTA_MODEL).to(DEVICE)
    model_inf.load_state_dict(torch.load(path, map_location=DEVICE))
    model_inf.eval()

    for template in TTA_TEMPLATES:
        tta_dataset = MCQDatasetTTA(test_df, roberta_tokenizer, template, MAX_LEN)
        tta_loader  = DataLoader(tta_dataset, batch_size=BATCH_EVAL, shuffle=False, num_workers=0)

        fold_preds = []
        with torch.no_grad():
            for batch in tqdm(tta_loader, desc=f'TTA [{template[:25]}...]'):
                ids  = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                fold_preds.append(model_inf(ids, mask).cpu().numpy())

        m2_logits += np.vstack(fold_preds) / total_passes

    del model_inf; gc.collect(); torch.cuda.empty_cache()

# Convert to probabilities via softmax (M4/M5 improvement over raw logits)
def softmax_np(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

m2_probs = softmax_np(m2_logits)  # (N, 5) probabilities

m2_test_preds = {}
for i, (_, row) in enumerate(test_df.iterrows()):
    m2_test_preds[str(int(row['id']))] = scores_to_top3(m2_probs[i])

print(f'Model 2 TTA inference done for {len(m2_test_preds)} questions')
print(f'Total passes: {total_passes} ({len(roberta_model_paths)} folds x {len(TTA_TEMPLATES)} templates)')

## 🧩 Section 6 — Model 3: LoRA ALBERT (Additional Pretrained Model)

> **Why ALBERT?** It uses parameter sharing across layers, making it much smaller and faster than RoBERTa.
> As a *different architecture* from RoBERTa, its errors and predictions are less correlated, improving ensemble diversity.

In [ ]:
ALBERT_MODEL = 'albert-base-v2'
albert_tokenizer = AutoTokenizer.from_pretrained(ALBERT_MODEL)

gc.collect(); torch.cuda.empty_cache()

# Single 80/20 split for ALBERT (faster, less GPU time needed)
tr_sub, val_sub = train_test_split(
    train_df, test_size=0.20, stratify=train_df['answer'], random_state=SEED
)

albert_tr_loader  = DataLoader(MCQDataset(tr_sub, albert_tokenizer, MAX_LEN), batch_size=BATCH_TRAIN, shuffle=True,  num_workers=0)
albert_val_loader = DataLoader(MCQDataset(val_sub, albert_tokenizer, MAX_LEN), batch_size=BATCH_EVAL,  shuffle=False, num_workers=0)

# Reuse same LoRAMCQModel class with ALBERT
albert_model = LoRAMCQModel(ALBERT_MODEL).to(DEVICE)
albert_opt = torch.optim.AdamW(
    [p for p in albert_model.parameters() if p.requires_grad],
    lr=3e-4, weight_decay=0.01
)

run3 = wandb.init(
    project='24f1002384-t22026',
    name='model3_lora_albert',
    config={'model': ALBERT_MODEL, 'lora_r': 16, 'lr': 3e-4, 'epochs': 3},
    reinit=True
)

best_albert_map3 = 0.0

for epoch in range(3):
    albert_model.train()
    ep_loss = 0.0
    for batch in tqdm(albert_tr_loader, desc=f'ALBERT Epoch {epoch+1}/3'):
        albert_opt.zero_grad()
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lbls = batch['labels'].to(DEVICE)

        logits = albert_model(ids, mask)
        loss = F.cross_entropy(logits, lbls)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(albert_model.parameters(), 1.0)
        albert_opt.step()
        ep_loss += loss.item()

    avg_loss = ep_loss / len(albert_tr_loader)

    albert_model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in albert_val_loader:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            logits = albert_model(ids, mask).cpu().numpy()
            for row_logits in logits:
                preds.append(scores_to_top3(row_logits))
            labels.extend([IDX2LABEL[l] for l in batch['labels'].numpy()])

    val_map = map_at_3(preds, labels)
    print(f'ALBERT Epoch {epoch+1} | Loss={avg_loss:.4f} | Val MAP@3={val_map:.4f}')
    wandb.log({'val_map3': val_map, 'loss': avg_loss})

    if val_map > best_albert_map3:
        best_albert_map3 = val_map
        torch.save(albert_model.state_dict(), 'albert_lora.pt')
        print(f'  💾 Saved ALBERT best (MAP@3={best_albert_map3:.4f})')

wandb.finish()
print(f'\n⭐ ALBERT Best Val MAP@3: {best_albert_map3:.4f}')

In [ ]:
# ALBERT test inference with TTA
albert_model.load_state_dict(torch.load('albert_lora.pt', map_location=DEVICE))
albert_model.eval()

m3_logits_acc = np.zeros((len(test_df), 5))

for template in TTA_TEMPLATES:
    tta_dataset_al = MCQDatasetTTA(test_df, albert_tokenizer, template, MAX_LEN)
    tta_loader_al  = DataLoader(tta_dataset_al, batch_size=BATCH_EVAL, shuffle=False, num_workers=0)

    fold_preds_al = []
    with torch.no_grad():
        for batch in tqdm(tta_loader_al, desc=f'ALBERT TTA [{template[:20]}...]'):
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            fold_preds_al.append(albert_model(ids, mask).cpu().numpy())

    m3_logits_acc += np.vstack(fold_preds_al) / len(TTA_TEMPLATES)

m3_probs = softmax_np(m3_logits_acc)  # (N, 5) probabilities

m3_test_preds = {}
for i, (_, row) in enumerate(test_df.iterrows()):
    m3_test_preds[str(int(row['id']))] = scores_to_top3(m3_probs[i])

del albert_model; gc.collect(); torch.cuda.empty_cache()
print(f'Model 3 (ALBERT TTA) inference done for {len(m3_test_preds)} questions')

## 🤝 Section 7 — Weighted Borda Count Ensemble + Lookup Overrides

In [ ]:
# Weighted Probability Averaging Ensemble (M5 improvement over Borda count)
# - Borda count only uses rank order, discarding confidence information
# - Probability averaging uses full softmax distributions: more mathematically sound
#
# BM25 Model 1: convert top-3 list to pseudo-probability via rank weights
# RoBERTa (M2): softmax probabilities from TTA-averaged logits
# ALBERT (M3):  softmax probabilities from TTA-averaged logits

W1, W2, W3 = 0.10, 0.65, 0.25   # Model weights (sum = 1.0)
BORDA_RANK_WEIGHTS = [0.50, 0.30, 0.15, 0.04, 0.01]  # pseudo-prob for rank 1-5

def bm25_to_prob(top3_letters: list) -> np.ndarray:
    """Convert BM25 top-3 list to a pseudo-probability vector of length 5."""
    prob = np.zeros(5)
    for rank, letter in enumerate(top3_letters):
        prob[LABEL2IDX[letter]] = BORDA_RANK_WEIGHTS[rank]
    # Fill remaining with small weight
    for c in CHOICES:
        if LABEL2IDX[c] not in [LABEL2IDX[l] for l in top3_letters]:
            prob[LABEL2IDX[c]] = BORDA_RANK_WEIGHTS[3]
    prob = prob / prob.sum()  # normalize
    return prob

final_predictions = {}

for i, (_, row) in enumerate(test_df.iterrows()):
    tid     = str(int(row['id']))
    test_id = int(row['id'])

    # ── HARD OVERRIDE: fuzzy lookup >=95 similarity ─────────────────────
    if test_id in lookup_hard:
        correct_ans = lookup_hard[test_id]
        m2_ordered  = m2_test_preds.get(tid, CHOICES)
        remaining   = [c for c in m2_ordered if c != correct_ans]
        final_predictions[tid] = [correct_ans] + remaining[:2]
        continue

    # ── WEIGHTED PROBABILITY AVERAGING for remaining questions ───────────
    # Get probabilities for each model
    p1 = bm25_to_prob(m1_test_preds.get(tid, CHOICES[:3]))
    p2 = m2_probs[i]   # Already softmax probabilities from TTA
    p3 = m3_probs[i]   # Already softmax probabilities from TTA

    # Weighted average
    p_final = W1 * p1 + W2 * p2 + W3 * p3

    # Soft fuzzy boost: slightly increase prob for soft-lookup answer
    if test_id in lookup_soft:
        boost_idx = LABEL2IDX[lookup_soft[test_id]]
        p_final[boost_idx] += 0.08  # Additive boost
        p_final = p_final / p_final.sum()  # Renormalize

    final_predictions[tid] = scores_to_top3(p_final)

print(f'Ensemble predictions for {len(final_predictions)} test questions')
print(f'  Hard lookup overrides: {len(lookup_hard)}')
print(f'  Probability-averaged:  {len(test_df) - len(lookup_hard)}')

## ✅ Section 8 — Submission Verification & Export

In [ ]:
errors = []
test_ids = set(str(int(i)) for i in test_df['id'])

if set(final_predictions.keys()) != test_ids:
    errors.append(f'ID mismatch: expected {len(test_ids)}, got {len(final_predictions)}')

for tid, preds in final_predictions.items():
    if len(preds) != 3:
        errors.append(f'ID {tid} has {len(preds)} predictions (expected 3)')
    if len(set(preds)) != 3:
        errors.append(f'ID {tid} has duplicate predictions: {preds}')
    if not all(p in CHOICES for p in preds):
        errors.append(f'ID {tid} has invalid prediction: {preds}')

if not errors:
    print('✅ All verification checks PASSED')
    rows = []
    for idx in sorted(test_df['id'].tolist(), key=int):
        tid = str(int(idx))
        rows.append({'ID': idx, 'Prediction': ' '.join(final_predictions[tid])})

    sub_df = pd.DataFrame(rows)
    sub_df.to_csv('submission.csv', index=False)
    print(f'💾 submission.csv saved ({len(sub_df)} rows)')
    print(sub_df.head(10))
else:
    print(f'❌ {len(errors)} verification errors:')
    for e in errors[:10]:
        print(f'  ⚠️ {e}')

In [ ]:
# Summary and prediction distribution analysis
import os

if not os.path.exists('submission.csv'):
    print('ERROR: submission.csv not found. Check verification step above.')
else:
    sub_df = pd.read_csv('submission.csv')

    # Top-1 prediction distribution
    top1 = sub_df['Prediction'].str.split().str[0]
    print('Top-1 prediction distribution:')
    print(top1.value_counts().sort_index())

    plt.figure(figsize=(8, 4))
    top1.value_counts().sort_index().plot(kind='bar', color='steelblue', edgecolor='white')
    plt.title('Top-1 Submission Prediction Distribution')
    plt.xlabel('Predicted Option')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

    print('\nSummary:')
    print(f'  Hard lookup overrides: {len(lookup_hard)} ({100*len(lookup_hard)/len(test_df):.1f}%)')
    print(f'  Soft lookup boosts:    {len(lookup_soft)}')
    print(f'  Model-only decisions:  {len(test_df) - len(lookup_hard)}')
    print(f'  RoBERTa CV MAP@3:      {np.mean(roberta_cv_scores):.4f}')
    print(f'  ALBERT Val MAP@3:      {best_albert_map3:.4f}')